In [ ]:
# ============================================================
# Conditional flow-matching training
# ============================================================

RESIDUAL_STD = 0.13850028854681287

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    flow_model.parameters(),
    lr=5e-4,
)

N_EPOCHS = 5

train_losses = []
val_losses = []

best_val_loss = np.inf


for epoch in range(N_EPOCHS):

    # ========================================================
    # Training
    # ========================================================

    flow_model.train()

    running_loss = 0.0
    n_samples = 0


    for batch in train_loader:

        bilinear = batch["bilinear"].to(DEVICE)
        target = batch["target"].to(DEVICE)

        # Target fine-scale residual
        R1 = target - bilinear

        # Gaussian source residual
        R0 = (
            RESIDUAL_STD
            * torch.randn_like(R1)
        )

        # Random flow time
        t = torch.rand(
            R1.shape[0],
            device=DEVICE,
        )

        t_view = t.view(-1, 1, 1, 1)

        # Linear probability path
        Rt = (
            (1 - t_view) * R0
            + t_view * R1
        )

        # Exact velocity for linear interpolation
        target_velocity = R1 - R0

        pred_velocity = flow_model(
            Rt,
            bilinear,
            t,
        )

        loss = criterion(
            pred_velocity,
            target_velocity,
        )

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            flow_model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()


        batch_size = bilinear.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        n_samples += batch_size


    train_loss = running_loss / n_samples


    # ========================================================
    # Validation
    # ========================================================

    flow_model.eval()

    running_loss = 0.0
    n_samples = 0


    with torch.no_grad():

        for batch in val_loader:

            bilinear = batch["bilinear"].to(DEVICE)
            target = batch["target"].to(DEVICE)

            R1 = target - bilinear

            R0 = (
                RESIDUAL_STD
                * torch.randn_like(R1)
            )

            t = torch.rand(
                R1.shape[0],
                device=DEVICE,
            )

            t_view = t.view(-1, 1, 1, 1)

            Rt = (
                (1 - t_view) * R0
                + t_view * R1
            )

            target_velocity = R1 - R0

            pred_velocity = flow_model(
                Rt,
                bilinear,
                t,
            )

            loss = criterion(
                pred_velocity,
                target_velocity,
            )

            batch_size = bilinear.size(0)

            running_loss += (
                loss.item() * batch_size
            )

            n_samples += batch_size


    val_loss = running_loss / n_samples

    train_losses.append(train_loss)
    val_losses.append(val_loss)


    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "epoch": epoch,
                "model_state": flow_model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "residual_std": RESIDUAL_STD,
            },
            FLOW_CHECKPOINT_FILE,
        )


    print(
        f"Epoch {epoch + 1:02d}/{N_EPOCHS} | "
        f"Train: {train_loss:.6f} | "
        f"Validation: {val_loss:.6f}"
    )


print("\nBest validation loss:", best_val_loss)


def generate_residual(
    model,
    bilinear,
    residual_std,
    n_steps=30,
):
    """
    Generate one stochastic residual field using Euler integration
    of the learned conditional flow.
    """

    R = (
        residual_std
        * torch.randn_like(bilinear)
    )

    dt = 1.0 / n_steps


    with torch.no_grad():

        for step in range(n_steps):

            t = torch.full(
                (bilinear.shape[0],),
                step / n_steps,
                device=bilinear.device,
            )

            velocity = model(
                R,
                bilinear,
                t,
            )

            R = R + dt * velocity


    return R

N_MEMBERS = 5

generated_fields = []


for _ in range(N_MEMBERS):

    generated_residual = generate_residual(
        flow_model,
        bilinear,
        residual_std=RESIDUAL_STD,
        n_steps=30,
    )

    generated = (
        bilinear
        + generated_residual
    )

    generated_fields.append(
        generated.cpu().numpy()
    )


generated_fields = np.concatenate(
    generated_fields,
    axis=0,
)

print("Ensemble shape:", generated_fields.shape)